# Fechar Mês — Fechamento Financeiro Mensal

Consolida o faturamento do mês (`gold_fechamento_mensal`) e valida a completude da execução diária antes de declarar o fechamento como válido — consultando `observability.pipeline_runs` para confirmar que todos os dias úteis do mês (calendário do Financeiro, Seg-Sex) tiveram execução com sucesso.

Se algum dia estiver faltando, o fechamento é gravado como `fechamento_valido = false` e um alerta é disparado via `NotificadorTabela` — publicar número financeiro "fechado" sobre dado incompleto é exatamente o tipo de erro que a observabilidade existe para prevenir.

Referências: ADR-006 (orquestração, fechamento mensal planejado desde o início), ADR-007 (alertas), ADR-014 (Gold).

## Widgets

- `mes_referencia`: mês a fechar, formato AAAA-MM (ex.: "2026-07"). Obrigatório — sem padrão, para evitar fechar o mês errado por engano.

In [0]:
dbutils.widgets.text("mes_referencia", "", "Mês de referência (AAAA-MM)")
dbutils.widgets.text("mes_referencia", "", "Mês de referência (AAAA-MM) — opcional, sobrescreve o cálculo automático")
dbutils.widgets.text("data_execucao", "", "Data de execução (AAAA-MM-DD) — usada para calcular o mês anterior, se mes_referencia vazio")

In [0]:
from datetime import date, timedelta
import calendar

mes_referencia_str = dbutils.widgets.get("mes_referencia")

if not mes_referencia_str:
    data_execucao_str = dbutils.widgets.get("data_execucao")
    data_execucao = date.fromisoformat(data_execucao_str) if data_execucao_str else date.today()
    primeiro_dia_mes_atual = data_execucao.replace(day=1)
    ultimo_dia_mes_anterior = primeiro_dia_mes_atual - timedelta(days=1)
    mes_referencia_str = ultimo_dia_mes_anterior.strftime("%Y-%m")

ano, mes = map(int, mes_referencia_str.split("-"))
primeiro_dia = date(ano, mes, 1)
ultimo_dia_numero = calendar.monthrange(ano, mes)[1]
ultimo_dia = date(ano, mes, ultimo_dia_numero)

print(f"Fechando mês: {mes_referencia_str}")
print(f"Período: {primeiro_dia} a {ultimo_dia}")

## Validação de completude

Consulta pipeline_runs para confirmar que todos os dias úteis do Financeiro (Seg-Sex, o calendário mais restritivo entre os 4 sistemas) tiveram execução de gerar_dados com sucesso no período.

In [0]:
from pyspark.sql.functions import col

dias_uteis_esperados = []
dia_atual = primeiro_dia
while dia_atual <= ultimo_dia:
    if dia_atual.weekday() in {0, 1, 2, 3, 4}:
        dias_uteis_esperados.append(dia_atual)
    dia_atual += timedelta(days=1)

df_runs_financeiro = spark.table("poc_pulse_observability.observability.pipeline_runs").filter(
    (col("pipeline") == "gerar_dados")
    & (col("item") == "financeiro")
    & (col("status") == "sucesso")
    & (col("data_referencia").between(str(primeiro_dia), str(ultimo_dia)))
)

dias_com_execucao = set(row.data_referencia for row in df_runs_financeiro.select("data_referencia").distinct().collect())
dias_esperados_str = set(str(d) for d in dias_uteis_esperados)
dias_faltando = sorted(dias_esperados_str - dias_com_execucao)

fechamento_valido = len(dias_faltando) == 0

print(f"Dias úteis esperados: {len(dias_esperados_str)}")
print(f"Dias com execução encontrados: {len(dias_com_execucao & dias_esperados_str)}")
print(f"Dias faltando: {dias_faltando}")
print(f"Fechamento válido: {fechamento_valido}")

## Consolidação e gravação

Calcula o faturamento consolidado do mês (a partir de gold_reconciliacao_financeira, mesma tabela já usada no dashboard), grava gold_fechamento_mensal, e dispara alerta via NotificadorTabela se o fechamento não for válido.

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType
from delta.tables import DeltaTable
import json
from src.observabilidade.notificadores import NotificadorTabela

# Consolidação
df_faturas_mes = spark.table("poc_pulse_observability.gold.gold_reconciliacao_financeira").filter(
    (col("data_faturamento") >= primeiro_dia) & (col("data_faturamento") <= ultimo_dia)
)
df_contas = spark.table("poc_pulse_observability.silver.financeiro_contas_receber").select("fatura_id", "status_conta")
df_consolidado = df_faturas_mes.join(df_contas, "fatura_id", "left")

total_faturado = df_consolidado.agg({"valor_faturado": "sum"}).collect()[0][0] or 0.0
total_recebido = df_consolidado.filter(col("status_conta") == "recebido").agg({"valor_faturado": "sum"}).collect()[0][0] or 0.0
total_faturas = df_consolidado.count()
total_divergentes = df_consolidado.filter(col("divergente")).count()

registro = {
    "mes_referencia": mes_referencia_str,
    "total_faturado": float(total_faturado),
    "total_recebido": float(total_recebido),
    "total_em_aberto": float(total_faturado - total_recebido),
    "total_faturas": total_faturas,
    "total_divergentes": total_divergentes,
    "dias_faltando": json.dumps(dias_faltando),
    "fechamento_valido": fechamento_valido,
}
print(registro)

# Gravação — schema explícito, MERGE por mes_referencia (idempotente)
schema = StructType([
    StructField("mes_referencia", StringType(), False),
    StructField("total_faturado", DoubleType(), True),
    StructField("total_recebido", DoubleType(), True),
    StructField("total_em_aberto", DoubleType(), True),
    StructField("total_faturas", IntegerType(), True),
    StructField("total_divergentes", IntegerType(), True),
    StructField("dias_faltando", StringType(), True),
    StructField("fechamento_valido", BooleanType(), False),
])
df_registro = spark.createDataFrame([registro], schema=schema)

tabela_destino = "poc_pulse_observability.gold.gold_fechamento_mensal"
if not spark.catalog.tableExists(tabela_destino):
    df_registro.write.format("delta").saveAsTable(tabela_destino)
    print("Primeira carga de gold_fechamento_mensal.")
else:
    tabela_delta = DeltaTable.forName(spark, tabela_destino)
    (
        tabela_delta.alias("destino")
        .merge(df_registro.alias("novo"), "destino.mes_referencia = novo.mes_referencia")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("gold_fechamento_mensal atualizada via MERGE.")

# Alerta condicional
if not fechamento_valido:
    notificador = NotificadorTabela(spark=spark)
    notificador.notificar({
        "tipo_evento": "fechamento_invalido",
        "origem": "fechar_mes",
        "severidade": "alta",
        "mensagem": f"Fechamento de {mes_referencia_str} inválido — {len(dias_faltando)} dia(s) sem execução registrada",
        "detalhes": {"mes_referencia": mes_referencia_str, "dias_faltando": dias_faltando},
    })
    print("Alerta disparado.")

In [0]:
spark.table("poc_pulse_observability.observability.alertas").filter(
    "tipo_evento = 'fechamento_invalido'"
).show(truncate=False)